# Epileptic Seizure Detection — patient-level split (no leakage)

Run the cells from top to bottom. Every model is built here, so every parameter is visible and editable.
Optuna is optional: use `best_param(...)` then `build(**best)` (on), or type the parameters yourself (off).

## 0. Setup (Kaggle)
Upload the project files as a Kaggle **Dataset** (or attach the zip), then set `PROJECT_DIR` and `CSV_PATH` below.

In [ ]:
import sys, os

# ── EDIT THESE TWO LINES ─────────────────────────────────────────────
PROJECT_DIR = '/kaggle/working'                     # folder that contains base.py, models/, etc.
CSV_PATH    = '/kaggle/input/YOUR-DATASET/balanced_dataset8.csv'   # your CSV file
# ─────────────────────────────────────────────────────────────────────

sys.path.insert(0, PROJECT_DIR)
print('Project files found:', sorted(f for f in os.listdir(PROJECT_DIR) if f.endswith('.py')))
print('CSV exists         :', os.path.exists(CSV_PATH))

In [ ]:
# Only needed if a package is missing (Kaggle already has most of them).
# !pip install -q xgboost optuna tqdm

## 1. Imports

In [ ]:
from preprocessing import prepare_data, load_raw
from models import LogisticModel, XGBoostModel, MambaModel
from experiment import train, evaluate
from loso import run_loso
from annotation import load_raw_annotation
from report import report, compare, loso_report, loso_compare
from plots import plot, plot_compare, plot_loso

## 2. Data
`X`, `y`, `g` are dictionaries with keys `'train'`, `'validation'`, `'test'`.  
`g` = patient id of every row (used for patient-grouped tuning and per-patient scores).

In [ ]:
X, y, g = prepare_data(CSV_PATH)          # baseline: one global scaler fitted on the training patients

# To use per-patient normalisation for EVERYTHING below, use this line instead:
# X, y, g = prepare_data(CSV_PATH, per_patient_norm='interictal')

## 2b. Does per-patient normalisation help? (fast Logistic Regression test)
- `None`         one global scaler (baseline)
- `'all'`        each patient scaled with statistics from ALL of that patient's windows (no labels used, but the result depends on each patient's seizure fraction)
- `'interictal'` each patient scaled with statistics from that patient's seizure-free windows (assumes a seizure-free calibration recording for a new patient — state this in the paper)

The loop uses its own variable names (`Xn`, `yn`, `gn`), so `X`, `y`, `g` above are NOT changed.

In [ ]:
runs = []
for mode in [None, 'all', 'interictal']:
    Xn, yn, gn = prepare_data(CSV_PATH, per_patient_norm=mode)
    m = train(LogisticModel.build(C=0.1, max_iter=2000), Xn['train'], yn['train'])
    s = evaluate(m, Xn['validation'], yn['validation'], Xn['test'], yn['test'], gn['test'])
    s['name'] = f'LogReg | norm={mode}'
    runs.append(s)
compare(runs); plot_compare(runs)

## 3. Logistic Regression — WITHOUT Optuna (you choose the parameters)

In [ ]:
model = LogisticModel.build(C=0.1, max_iter=2000)     # edit any value here
model = train(model, X['train'], y['train'])
scores_lr = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
report(scores_lr); plot(scores_lr)

## 4. XGBoost — WITHOUT Optuna (all parameters typed by hand)

In [ ]:
model = XGBoostModel.build(n_estimators=500, max_depth=8, learning_rate=0.05,
                           subsample=0.9, colsample_bytree=0.6, min_child_weight=10,
                           gamma=0.0, reg_alpha=0.1, reg_lambda=1.0)
model = train(model, X['train'], y['train'])
scores_xgb = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
report(scores_xgb); plot(scores_xgb)

## 5. XGBoost — WITH Optuna
`best_param` prints the best parameters; `build(**best)` uses them. Start with a small `n_trials` to check it runs.

In [ ]:
best  = XGBoostModel.best_param(X['train'], y['train'], g['train'], n_trials=10)
model = XGBoostModel.build(**best)
model = train(model, X['train'], y['train'])
scores_xgb_tuned = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
scores_xgb_tuned['name'] = 'XGBoost (Optuna)'
report(scores_xgb_tuned); plot(scores_xgb_tuned)

## 6. Mamba (optional — needs PyTorch; skip this section if you do not use it)
The 22 EEG channels form the sequence. State this in the paper (channel-axis modelling; the CSV has no time order).

In [ ]:
# WITHOUT Optuna:
model = MambaModel.build(d_model=64, n_layers=2, d_state=16, dropout=0.2,
                         lr=1e-3, weight_decay=1e-2, epochs=30)
model = train(model, X['train'], y['train'])
scores_mamba = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
report(scores_mamba); plot(scores_mamba)

In [ ]:
# WITH Optuna (slow — start with n_trials=3). Tuning runs 10 epochs; final training uses 30.
best  = MambaModel.best_param(X['train'], y['train'], g['train'], n_trials=3)
model = MambaModel.build(**{**best, 'epochs': 30})
model = train(model, X['train'], y['train'])
scores_mamba_tuned = evaluate(model, X['validation'], y['validation'], X['test'], y['test'], g['test'])
scores_mamba_tuned['name'] = 'Mamba (Optuna)'
report(scores_mamba_tuned)

## 7. Compare all models
Only list models you actually ran.

In [ ]:
all_scores = [scores_lr, scores_xgb, scores_xgb_tuned]          # add scores_mamba if you ran it
compare(all_scores); plot_compare(all_scores)

## 8. Wrong-parameter check (optional demo)
Each model rejects parameters that belong to another model.

In [ ]:
try:
    XGBoostModel.build(d_model=64)          # d_model is a Mamba parameter
except ValueError as e:
    print(e)

## 9. Leave-One-Subject-Out (LOSO)
Every real patient (20 after merging chb17a/b/c and chb21 into chb17 / chb01) is held out once.
In every fold the imputer, scaler, model and threshold are fitted **without** that patient.

**Hyperparameters:** use fixed values you chose *before* looking at results (e.g. the model defaults), or `tune=True` (Optuna inside every fold, very slow).
Do **not** reuse the Optuna values found on the fixed split: those were tuned on patients that are held out in some LOSO folds.

In [ ]:
raw = load_raw(CSV_PATH)          # read the CSV once  (or: load_raw(CSV_PATH, per_patient_norm='interictal'))

### 9a. Quick check on 2 held-out patients (fast model)

In [ ]:
quick = run_loso(raw, LogisticModel, params=dict(C=0.1, max_iter=2000), hold_out=['chb16', 'chb17'])
loso_report(quick)

### 9b. Full LOSO — all 20 patients
About one model fit per patient (with XGBoost defaults roughly an hour in total). Try `hold_out=['chb16', 'chb17']` first.

In [ ]:
res_xgb = run_loso(raw, XGBoostModel, params={})          # params={} = XGBoost defaults, not tuned
loso_report(res_xgb); plot_loso(res_xgb)

### 9c. SNN ablation under LOSO (same classifier, same folds, paired comparison)
Three full LOSO runs. Set `MODEL, PARAMS` to a fast model to test first.

In [ ]:
MODEL, PARAMS = XGBoostModel, {}                     # or: LogisticModel, dict(C=0.1, max_iter=2000)
runs = [run_loso(raw, MODEL, params=PARAMS, features=f, label=f'{MODEL.name} | {f}')
        for f in ['conventional', 'snn', 'all']]
loso_compare(runs, baseline=0)                       # baseline = conventional features only

## 10. Annotation extension test (R1-7)
The manuscript extends the positive class by 10s before onset and 5s after offset. Reviewer 1 asked
whether this inflates performance versus using only the true seizure interval.

Needs a CSV with an `in_core_ictal` column. `definition='core'` drops the pre/post-ictal-only windows
entirely (they are neither its positive nor its negative class) and uses `in_core_ictal` as the label
instead of `label`. Everything else (split, imputer, scaler, model, threshold) is identical to Section 9.

In [ ]:
r_ext  = load_raw_annotation(CSV_PATH, definition='extended')   # current manuscript definition (label==1)
r_core = load_raw_annotation(CSV_PATH, definition='core')       # true ictal interval only (in_core_ictal)

### 10a. Quick check on 2 patients

In [ ]:
q_ext  = run_loso(r_ext,  LogisticModel, params=dict(C=0.1, max_iter=2000), hold_out=['chb16', 'chb17'], label='extended')
q_core = run_loso(r_core, LogisticModel, params=dict(C=0.1, max_iter=2000), hold_out=['chb16', 'chb17'], label='core-ictal only')
loso_compare([q_ext, q_core], baseline=0)

### 10b. Full LOSO, both definitions (paired per-patient AUC comparison + Wilcoxon)

In [ ]:
res_ext  = run_loso(r_ext,  XGBoostModel, params={}, label='extended (label==1)')
res_core = run_loso(r_core, XGBoostModel, params={}, label='core-ictal only')
loso_compare([res_ext, res_core], baseline=0)